# gradient-routing-slm: one run on a Kaggle T4
Settings: **Accelerator = GPU T4** (x1 or x2, only one is used), **Internet = On**. Edit the first cell, then Run All.
The last cell writes `/kaggle/working/<mode>_s<seed>.zip`. Unzip it into `results/<mode>_s<seed>/` in the repo.

In [ ]:
ROUTING_MODE = "token"   # dense (baseline), batch (v0), token (v1), label (oracle baseline)
SEED = 0                 # model init and batch order; the dataset and val sets stay fixed
MAX_ITERS = 5000
REPO_URL = "https://github.com/sandwichhlord/gradient-routing-slm.git"

In [ ]:
!pip install -q -U tiktoken datasets

In [ ]:
!rm -rf /tmp/repo && git clone --depth 1 {REPO_URL} /tmp/repo
%cd /tmp/repo
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import json, os, shutil, subprocess
import torch
assert torch.cuda.is_available(), "No GPU: set Accelerator = GPU T4 in the notebook settings"
RUN = f"{ROUTING_MODE}_s{SEED}"
OUT = f"out_{RUN}"
env = {**os.environ, "ROUTING_MODE": ROUTING_MODE, "SEED": str(SEED), "MAX_ITERS": str(MAX_ITERS)}
p = subprocess.Popen(["bash", "run_all.sh"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
with open("run.log", "w") as log:
    for line in p.stdout:
        print(line, end="")
        log.write(line)
rc = p.wait()
os.makedirs(OUT, exist_ok=True)
shutil.copy("run.log", f"{OUT}/run.log")
if os.path.exists("data/out/meta.json"):
    shutil.copy("data/out/meta.json", f"{OUT}/data_meta.json")
    mode = json.load(open("data/out/meta.json"))["mode"]
    print()
    print("DATA MODE:", mode, "" if mode == "real" else " <-- WARNING: HF download failed, results are on synthetic data!")
print("run_all.sh exit code:", rc, "(the last cell still zips whatever was produced)")

In [ ]:
from IPython.display import Image, display
for name in ["loss_curves.png", "ablation_heatmap.png", "routing_by_domain.png"]:
    if os.path.exists(f"{OUT}/{name}"):
        display(Image(f"{OUT}/{name}"))
for name in ["ablation_summary.txt", "samples.txt"]:
    if os.path.exists(f"{OUT}/{name}"):
        print(open(f"{OUT}/{name}").read())

In [ ]:
zip_path = shutil.make_archive(f"/kaggle/working/{RUN}", "zip", OUT)
print("wrote", zip_path, f"({os.path.getsize(zip_path) / 1e6:.0f} MB)")
assert rc == 0, f"run_all.sh failed with exit code {rc}: see run.log inside the zip"